Load Dataset from DeepLoc


In [2]:
import pandas as pd

df = pd.read_csv("https://services.healthtech.dtu.dk/services/DeepLoc-2.0/data/Swissprot_Train_Validation_dataset.csv")

In [3]:
df.head()

,Unnamed: 0,ACC,Kingdom,Partition,Membrane,Cytoplasm,Nucleus,Extracellular,Cell membrane,Mitochondrion,Plastid,Endoplasmic reticulum,Lysosome/Vacuole,Golgi apparatus,Peroxisome,Sequence
0,0,Q28165,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,1,Q86U42,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,2,Q0GA42,Metazoa,3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,3,P82349,Metazoa,1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,4,Q7L5N1,Metazoa,1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


Extract Cytoplasm Presence Data

In [4]:
import numpy as np

y_data = df["Cytoplasm"].to_list()
y = 5
print(f"First {y} Entries: {y_data[:y]}")

First 5 Entries: [1.0, 1.0, 0.0, 1.0, 1.0]


Extract Protein Sequences

In [5]:
x_data = df["Sequence"].to_list()
x = 4
print(f"{x}th Entries: {x_data[x]}")

4th Entries: MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPLVILNISDHWIRMRSQEGRPVQVIGALIGKQEGRNIEVMNSFELLSHTVEEKIIIDKEYYYTKEEQFKQVFKELEFLGWYTTGGPPDPSDIHVHKQVCEIIESPLFLKLNPMTKHTDLPVSVFESVIDIINGEATMLFAELTYTLATEEAERIGVDHVARMTATGSGENSTVAEHLIAQHSAIKMLHSRVKLILEYVKASEAGEVPFNHEILREAYALCHCLPVLSTDKFKTDFYDQCNDVGLMAYLGTITKTCNTMNQFVNKFNVLYDRQGIGRRMRGLFF


Extract Features: Amino acid normalized frequency count and sequence length

In [14]:
amino_acids = np.array(["A", "R", "N", "D", "C", "Q", "E", "G", "H", "I","L", "K", "M", "F", "P","S", "T", "W","Y","V"])

features = []
for seq in x_data: 
    if seq == " ":
        continue
    seq_length = len(seq)
    feature_vector = [seq_length]
    for amino in amino_acids:
        feature_vector.append(seq.count(amino)/seq_length)
    features.append(feature_vector)


print(f"First Entry: {features[:2]}")

First Entry: [[306, 0.11437908496732026, 0.0915032679738562, 0.03594771241830065, 0.032679738562091505, 0.006535947712418301, 0.013071895424836602, 0.13398692810457516, 0.13071895424836602, 0.013071895424836602, 0.032679738562091505, 0.049019607843137254, 0.032679738562091505, 0.0196078431372549, 0.026143790849673203, 0.09477124183006536, 0.06862745098039216, 0.029411764705882353, 0.0032679738562091504, 0.032679738562091505, 0.0392156862745098], [306, 0.11437908496732026, 0.0915032679738562, 0.032679738562091505, 0.032679738562091505, 0.006535947712418301, 0.013071895424836602, 0.13398692810457516, 0.13071895424836602, 0.013071895424836602, 0.032679738562091505, 0.049019607843137254, 0.032679738562091505, 0.0196078431372549, 0.026143790849673203, 0.09803921568627451, 0.06862745098039216, 0.029411764705882353, 0.0032679738562091504, 0.032679738562091505, 0.0392156862745098]]


In [19]:
nsamples =  len(x_data)
indices = np.arange(nsamples)
np.random.shuffle(indices)

train_split = int(nsamples*0.7)
test_split = int(nsamples*0.2) + train_split
train_idx, test_idx, validation_idx = indices[:train_split], indices[train_split:test_split], indices[test_split:]

print(f"train, test: {train_idx}")

train, test: [24723  6626 20811 ... 14443 11274 14654]


Data splitting for Testing (70%), Training (20%), and Validation (10%)

In [26]:

X = np.array(features)
y = np.array(y_data)

X_train, y_train = X[train_idx], y[train_idx]

X_test, y_test = X[test_idx], y[test_idx]
    
X_valid, y_valid = X[validation_idx], y[validation_idx]


print(X.shape)
print(X_train.shape)
print(X_test.shape)
print(X_valid.shape)
print(f"Y train_shape: {y_train.shape}")

(28303, 21)
(19812, 21)
(5660, 21)
(2831, 21)
Y train_shape: (19812,)


Standardization using Z-score

In [23]:
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)
std[std == 0] = 1

X_train = (X_train - mean)/std
print(X_train)
X_test = (X_test - mean)/std

X_valid = (X_valid - mean)/std


[[-0.53487648  0.90072528  0.31927256 ... -0.42085568  0.84066905
  -0.47088557]
 [ 0.13370503  1.36284428  1.44775719 ... -0.10128932 -1.09413833
  -0.72364877]
 [-0.64347856 -0.13431205  0.80964391 ... -0.72500263  1.48973588
  -0.04870563]
 ...
 [-0.54505793 -1.21645525 -0.37689272 ...  1.05449058  1.82245081
   1.63980788]
 [-0.32445997  0.5247835  -0.87390717 ...  0.51278882 -0.93592693
  -1.29208746]
 [-0.02071354  0.93079451  0.01671254 ...  0.31942761 -0.13557407
   0.59892354]]


In [32]:
import importlib
import protein_classifier

importlib.reload(protein_classifier)
from protein_classifier import ProteinClassifier

model = ProteinClassifier(x_train=X_train, y_train=y_train, x_test=X_test, y_test=y_test, x_valid=X_valid, y_valid=y_valid)

model.train_cross_entropy()

Weights: [-0.17107361 -0.03215645 -0.0193333  -0.01482279 -0.00866301 -0.0163161
 -0.00379961 -0.00048902 -0.03460066 -0.00650712 -0.02237204 -0.0431215
 -0.01688596 -0.01047206 -0.02549074 -0.01797711 -0.03442269 -0.02660117
 -0.01011028 -0.01553374 -0.0298755 ]
